# ALS tuning - effect of k and lambda

Runs ALS for a grid of k (rank) and lambda (regParam) values on
MovieLens 100k (official `ua.base` / `ua.test` split) and records one result row
per run in the shared CSV. Use it to see how k and lambda change the test RMSE.
Fixed number of cores.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "als").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "als" / "src"))

from datasets import ensure_movielens
from spark_utils import get_spark_session, stop_spark
from split_utils import load_ml100k_predefined, clean_cold_start
from als_utils import run_als
from results_utils import append_result, new_run_id

In [2]:

DATASET       = "ml-100k"
RESULTS_CSV   = str(PROJECT_ROOT / "results" / "als_results.csv")

K_VALUES      = [5, 10, 20, 50]
LAMBDA_VALUES = [0.05, 0.1, 0.15]
MAX_ITER      = 10
SEED          = 42
NUM_CORES     = 4

DATA_DIR = str(ensure_movielens(DATASET, PROJECT_ROOT / "data"))


## Grid runs
One Spark session loops over every (k, lambda) pair

In [3]:
spark = get_spark_session(f"als_{DATASET}_tuning", num_cores=NUM_CORES)

train, test = load_ml100k_predefined(spark, DATA_DIR)
test_clean = clean_cold_start(train, test).cache()
train = train.cache()
train_size = train.count()
test_size = test_clean.count()
dataset_size = train_size + test_size

for k in K_VALUES:
    for lam in LAMBDA_VALUES:
        row = run_als(
            train, test_clean,
            dataset=DATASET, dataset_size=dataset_size, num_cores=NUM_CORES,
            k=k, lam=lam, max_iter=MAX_ITER, seed=SEED,
            warmup=False, run_id=new_run_id(),
        )
        append_result(row, RESULTS_CSV)
        print(f"  k={k:>3}  lambda={lam:<5}  RMSE={row['test_rmse']:.4f}  train_time={row['train_time']:.2f}s")

stop_spark(spark)
print("Results appended to", RESULTS_CSV)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 15:20:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


  k=  5  lambda=0.05   RMSE=0.9744  train_time=1.85s
  k=  5  lambda=0.1    RMSE=0.9483  train_time=0.84s
  k=  5  lambda=0.15   RMSE=0.9480  train_time=0.68s
  k= 10  lambda=0.05   RMSE=0.9953  train_time=0.70s
  k= 10  lambda=0.1    RMSE=0.9517  train_time=0.60s
  k= 10  lambda=0.15   RMSE=0.9500  train_time=0.58s
  k= 20  lambda=0.05   RMSE=1.0058  train_time=0.68s
  k= 20  lambda=0.1    RMSE=0.9545  train_time=0.63s
  k= 20  lambda=0.15   RMSE=0.9511  train_time=0.60s
  k= 50  lambda=0.05   RMSE=0.9844  train_time=1.22s
  k= 50  lambda=0.1    RMSE=0.9510  train_time=1.16s
  k= 50  lambda=0.15   RMSE=0.9514  train_time=1.20s
Results appended to /mnt/c/Users/89526/Documents/GitHub/ccdpp-pyspark-als-movielens/results/als_results.csv


#RMSE by k and lambda

In [4]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
grid = df[(df.dataset == DATASET) & (df.k.isin(K_VALUES)) & (df["lambda"].isin(LAMBDA_VALUES))]
table = grid.pivot_table(index="k", columns="lambda", values="test_rmse", aggfunc="min")
print("test RMSE (rows = k, columns = lambda):")
table

test RMSE (rows = k, columns = lambda):


lambda,0.05,0.10,0.15
k,,,
5,0.974445,0.948341,0.947985
10,0.995310,0.951691,0.949984
20,1.005791,0.954541,0.951068
50,0.984438,0.951026,0.951381
